# V6 research-backed advanced fraud challengers

Read-only retrospective experiment: weighted XGBoost, balanced random forest, normal-behavior Isolation Forest, and fixed rank blending. A paired feature ablation tests seven new strictly prior product/country/amount features. A shuffled-label negative control tests whether fitting can obtain similar exploratory selection results without a genuine label-feature relationship.

The 2026 final test stays sealed. Fit sampling uses the same 10% negative hash sample as V5 and all positives. Evaluation retains natural prevalence. No source/scoring tables, UC functions, model registrations or serving endpoints are created. The label and feature availability at authorization remain unverified.


In [ ]:
%run ./train_v4

In [ ]:
%run ./train_v5

In [ ]:
%run ./train_v6

In [ ]:
from datetime import datetime, timedelta
T0 = datetime(2024,1,2,13)
base = dict(product_id="p",customer_id="c",is_fraud=False,amount=80.0,currency="USD",amount_usd=None,transaction_type="Payment",channel="App",transaction_country="MX",merchant_category=None,transaction_category="Purchase",merchant_name="M",latitude=19.4,longitude=-99.1)
records = [dict(base,transaction_id=f"before_{i}",transaction_date=T0-timedelta(minutes=5-i),amount=float((i+1)*10),transaction_country="CO") for i in range(5)]
records += [dict(base,transaction_id="mixed_currency",transaction_date=T0-timedelta(minutes=2),currency="COP",amount=999.0,transaction_country="CO")]
records += [dict(base,transaction_id=n,transaction_date=t) for n,t in [("same_a",T0),("same_b",T0),("sealed",datetime(2026,1,1))]]
schema = "transaction_id string, product_id string, customer_id string, transaction_date timestamp, is_fraud boolean, amount double, currency string, amount_usd double, transaction_type string, channel string, transaction_country string, merchant_category string, transaction_category string, merchant_name string, latitude double, longitude double"
source = spark.createDataFrame(records,schema)
observed = build_v6_features(spark, build_feature_frame, source_frame=source).select("transaction_id",*V6_EXTRA_NUMERIC).collect()
assert len(observed)==8
for row in observed:
 if row["transaction_id"].startswith("same"):
  expected = causal_extra_features(dict(base,transaction_date=T0),records)
  for name in V6_EXTRA_NUMERIC:
   assert abs(float(row[name])-float(expected[name]))<1e-6,(name,row[name],expected[name])
print("V6 Spark fixture passed: same-time/future exclusion, lower bound, mixed currency, product/customer windows.")


In [ ]:
import numpy as np
from xgboost import XGBClassifier
from imblearn.ensemble import BalancedRandomForestClassifier
from sklearn.ensemble import IsolationForest
xx=np.random.default_rng(42).normal(size=(120,4)).astype(np.float32)
yy=np.r_[np.ones(10),np.zeros(90),np.ones(5),np.zeros(15)].astype(int)
mm=XGBClassifier(n_estimators=5,max_depth=2,tree_method="hist",eval_metric="aucpr",early_stopping_rounds=2,n_jobs=2).fit(xx[:100],yy[:100],eval_set=[(xx[100:],yy[100:])],verbose=False)
bb=BalancedRandomForestClassifier(n_estimators=2,max_depth=2,sampling_strategy=0.2,replacement=True,bootstrap=False,n_jobs=2,random_state=42).fit(xx[:100],yy[:100])
ii=IsolationForest(n_estimators=2,max_samples=16,random_state=42).fit(xx[:100][yy[:100]==0])
assert mm.predict_proba(xx[100:]).shape==(20,2) and bb.predict_proba(xx[100:]).shape==(20,2) and len(ii.score_samples(xx[100:]))==20
print("XGBoost / balanced forest / novelty runtime smoke passed; toy outcomes are not performance evidence.")


## Research and protocol

The ULB [fraud handbook ensemble comparison](https://fraud-detection-handbook.github.io/fraud-detection-handbook/Chapter_6_ImbalancedLearning/Ensembling.html) studies balanced forests and weighted boosting. [Balanced forest documentation](https://imbalanced-learn.org/stable/references/generated/imblearn.ensemble.BalancedRandomForestClassifier.html) defines per-tree class resampling. These are techniques to test; external reported percentages are not transferred to this dataset.

Fit before 2025-01-01; select candidates/early stopping in January–March; choose thresholds in April–June; compare on already inspected July–December. These periods do not erase earlier experiment exposure. The rank blend has fixed equal weights and normalization references from fit-only negatives, not future evaluation distributions. High-precision scenarios require at least 100 alerts, 20% recall, and at most 1% alert rate. No-alert precision stays undefined.


In [ ]:
import json
import mlflow
result=run_v6(spark,mlflow,build_feature_frame,NUMERIC_FEATURES,CATEGORICAL_FEATURES,prediction_metrics,operating_points,available_memory_bytes)
print(json.dumps(result,indent=2,allow_nan=False))
dbutils.notebook.exit(json.dumps(result,allow_nan=False))

## Recorded execution: 2026-10-04

One-time run `482234883980497` completed successfully. A1 enhanced XGBoost was selected on January–March average precision (0.00108108), ahead of base XGBoost (0.00104737). The shuffled-label control achieved AP 0.00097706; this single conditional control does not establish label randomness.

Later July–December ROC-AUC was 0.5023 and average precision 0.00098223. At the frozen approximately 1% review threshold, 7,536 alerts contained 8 fraud labels and 7,528 false positives: **0.106% precision, 1.144% recall**. V5 found 12 labels in 7,468 alerts (0.161% precision). None of the supported 20%/50%/80% precision targets was feasible in the designated operating period.

**Not promoted.** The seven new features, balanced forest, anomaly ranking, and fixed blend did not establish useful later-period discrimination. Only the selected candidate was evaluated later. Source tables were read only and final-test rows remained sealed. Next investigate digital-event coverage/timing after SELECT is granted and obtain label/score provenance, rather than rerun this notebook without a changed hypothesis.

See the repository report `ml/reports/2026-10-04/training_v6_advanced_report.md` and aggregate JSON evidence for complete denominators, monthly results, exact settings, and prerequisites. Notebook outputs stay empty; recorded findings are documentation of the dated run, not results from a future rerun.
